In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
# Load the dataset
PREPROCESSED_PATH = os.path.join('D:\\', 'corruption')
#PREPROCESSED_PATH = os.path.join('..', '..', '..', 'static', 'preprocessed_datasets', 'corruption')

file_path = os.path.join(PREPROCESSED_PATH, 'cleaned_corruption_data.csv')
data = pd.read_csv(file_path, delimiter=',')

display(data)

,Country,QB1,QB10,QB15.1,QB15.10,QB15.11,QB15.12,QB15.13,QB15.2,QB15.3,...,CPI 2023,df_totalscore,df_fulfillment,dj_total score,dj_fulfillment,tindex_total,tindex_fulfillment,Latitude,Longitude,Country_ID
0,AT,0.690,0.290,0.62,0.39,0.50,0.60,0.46,0.62,0.56,...,71.000,9.000,0.643,5.000,0.833,14.000,0.700,48.208,16.374,0
1,BE,0.800,0.330,0.70,0.36,0.59,0.59,0.44,0.68,0.62,...,73.000,8.000,0.571,5.000,0.833,13.000,0.650,50.850,4.352,1
2,BG,0.700,0.310,0.79,0.16,0.78,0.81,0.18,0.84,0.78,...,45.000,11.000,0.786,6.000,1.000,17.000,0.850,42.698,23.322,2
3,CY,0.690,0.460,0.88,0.25,0.87,0.81,0.31,0.89,0.93,...,53.000,4.000,0.286,5.000,0.833,9.000,0.450,35.126,33.430,3
4,CZ,0.730,0.410,0.80,0.28,0.56,0.64,0.34,0.84,0.73,...,64.111,9.500,0.679,6.000,1.000,15.500,0.775,50.076,14.438,4
5,D-E,0.820,0.370,0.61,0.28,0.49,0.55,0.30,0.64,0.54,...,64.111,10.037,0.717,5.741,0.957,15.778,0.789,51.166,10.452,5
6,D-W,0.820,0.420,0.58,0.28,0.30,0.39,0.33,0.57,0.40,...,64.111,10.037,0.717,5.741,0.957,15.778,0.789,51.166,10.452,6
7,DE,0.820,0.410,0.59,0.28,0.34,0.42,0.33,0.59,0.43,...,78.000,9.500,0.679,6.000,1.000,15.500,0.775,51.166,10.452,7
8,DK,0.860,0.430,0.48,0.34,0.19,0.24,0.47,0.53,0.31,...,90.000,9.000,0.643,6.000,1.000,15.000,0.750,55.676,12.568,8
9,EE,0.770,0.440,0.62,0.19,0.43,0.50,0.29,0.60,0.40,...,76.000,13.000,0.929,6.000,1.000,19.000,0.950,59.437,24.754,9


In [24]:
# 1. QB_avg: average of all QB columns
qb_columns = [col for col in data.columns if col.startswith('QB')]
if qb_columns:
    data['QB_avg'] = data[qb_columns].apply(lambda row: sum(row) / len(row), axis=1)
#display(data)

# print(qb_columns)

# 2. QB6_QB13: product of QB6 and QB13
if 'QB6' in data.columns and 'QB13' in data.columns:
    data['QB6_QB13'] = data['QB6'] * data['QB13']

display(data)

# 3. QB15.6_QB15.7: product of QB15.6 and QB15.7
if 'QB15.6' in data.columns and 'QB15.7' in data.columns:
    data['QB15.6_QB15.7'] = data['QB15.6'] * data['QB15.7']

,Country,QB1,QB10,QB15.1,QB15.10,QB15.11,QB15.12,QB15.13,QB15.2,QB15.3,...,dj_total score,dj_fulfillment,tindex_total,tindex_fulfillment,Latitude,Longitude,Country_ID,QB_avg,QB6_QB13,QB15.6_QB15.7
0,AT,0.690,0.290,0.62,0.39,0.50,0.60,0.46,0.62,0.56,...,5.000,0.833,14.000,0.700,48.208,16.374,0,0.411544,0.0800,0.23
1,BE,0.800,0.330,0.70,0.36,0.59,0.59,0.44,0.68,0.62,...,5.000,0.833,13.000,0.650,50.850,4.352,1,0.418517,0.1036,0.44
2,BG,0.700,0.310,0.79,0.16,0.78,0.81,0.18,0.84,0.78,...,6.000,1.000,17.000,0.850,42.698,23.322,2,0.446714,0.0561,0.68
3,CY,0.690,0.460,0.88,0.25,0.87,0.81,0.31,0.89,0.93,...,5.000,0.833,9.000,0.450,35.126,33.430,3,0.503731,0.1656,0.58
4,CZ,0.730,0.410,0.80,0.28,0.56,0.64,0.34,0.84,0.73,...,6.000,1.000,15.500,0.775,50.076,14.438,4,0.443466,0.0480,0.45
5,D-E,0.820,0.370,0.61,0.28,0.49,0.55,0.30,0.64,0.54,...,5.741,0.957,15.778,0.789,51.166,10.452,5,0.366785,0.0616,0.37
6,D-W,0.820,0.420,0.58,0.28,0.30,0.39,0.33,0.57,0.40,...,5.741,0.957,15.778,0.789,51.166,10.452,6,0.329943,0.0832,0.28
7,DE,0.820,0.410,0.59,0.28,0.34,0.42,0.33,0.59,0.43,...,6.000,1.000,15.500,0.775,51.166,10.452,7,0.337939,0.0805,0.29
8,DK,0.860,0.430,0.48,0.34,0.19,0.24,0.47,0.53,0.31,...,6.000,1.000,15.000,0.750,55.676,12.568,8,0.294092,0.1848,0.17
9,EE,0.770,0.440,0.62,0.19,0.43,0.50,0.29,0.60,0.40,...,6.000,1.000,19.000,0.950,59.437,24.754,9,0.336083,0.0336,0.25


In [25]:
# 1. QB_avg: average of all QB columns
qb_columns = [col for col in data.columns if col.startswith('QB')]
if qb_columns:
    data['QB_avg'] = data[qb_columns].apply(lambda row: sum(row) / len(row), axis=1)
#display(data)

# print(qb_columns)

# 2. QB6_QB13: product of QB6 and QB13
if 'QB6' in data.columns and 'QB13' in data.columns:
    data['QB6_QB13'] = data['QB6'] * data['QB13']

display(data)

# 3. QB15.6_QB15.7: product of QB15.6 and QB15.7
if 'QB15.6' in data.columns and 'QB15.7' in data.columns:
    data['QB15.6_QB15.7'] = data['QB15.6'] * data['QB15.7']

# 4. CPI_avg: average of CPI 2020-2023
cpi_columns = ['CPI 2020', 'CPI 2021', 'CPI 2022', 'CPI 2023']
if all(col in data.columns for col in cpi_columns):
    data['CPI_avg'] = data[cpi_columns].mean(axis=1)

# 5. CPI_year_minus_avg: differences between each CPI and the average
for year in ['2020', '2021', '2022', '2023']:
    col_name = f'CPI_{year}_minus_avg'
    if f'CPI {year}' in data.columns and 'CPI_avg' in data.columns:
        data[col_name] = data[f'CPI {year}'] - data['CPI_avg']

# 6. CPI_percent_change: percentage change between CPI 2023 and CPI 2020
if 'CPI 2020' in data.columns and 'CPI 2023' in data.columns:
    data['CPI_percent_change'] = (data['CPI 2023'] - data['CPI 2020']) / data['CPI 2020']

# 7. CPI_category: categories based on CPI_avg
if 'CPI_avg' in data.columns:
    data['CPI_category'] = pd.cut(data['CPI_avg'], bins=[0, 50, 70, float('inf')], 
                                  labels=['Low', 'Medium', 'High'], right=False)

# 8. CPI_category_encoded: encoding of CPI_category
if 'CPI_category' in data.columns:
    category_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
    data['CPI_category_encoded'] = data['CPI_category'].map(category_mapping)



,Country,QB1,QB10,QB15.1,QB15.10,QB15.11,QB15.12,QB15.13,QB15.2,QB15.3,...,dj_total score,dj_fulfillment,tindex_total,tindex_fulfillment,Latitude,Longitude,Country_ID,QB_avg,QB6_QB13,QB15.6_QB15.7
0,AT,0.690,0.290,0.62,0.39,0.50,0.60,0.46,0.62,0.56,...,5.000,0.833,14.000,0.700,48.208,16.374,0,0.408976,0.0800,0.1938
1,BE,0.800,0.330,0.70,0.36,0.59,0.59,0.44,0.68,0.62,...,5.000,0.833,13.000,0.650,50.850,4.352,1,0.409397,0.1036,0.2016
2,BG,0.700,0.310,0.79,0.16,0.78,0.81,0.18,0.84,0.78,...,6.000,1.000,17.000,0.850,42.698,23.322,2,0.419660,0.0561,0.0780
3,CY,0.690,0.460,0.88,0.25,0.87,0.81,0.31,0.89,0.93,...,5.000,0.833,9.000,0.450,35.126,33.430,3,0.484697,0.1656,0.1275
4,CZ,0.730,0.410,0.80,0.28,0.56,0.64,0.34,0.84,0.73,...,6.000,1.000,15.500,0.775,50.076,14.438,4,0.428262,0.0480,0.1216
5,D-E,0.820,0.370,0.61,0.28,0.49,0.55,0.30,0.64,0.54,...,5.741,0.957,15.778,0.789,51.166,10.452,5,0.355733,0.0616,0.1464
6,D-W,0.820,0.420,0.58,0.28,0.30,0.39,0.33,0.57,0.40,...,5.741,0.957,15.778,0.789,51.166,10.452,6,0.323720,0.0832,0.1173
7,DE,0.820,0.410,0.59,0.28,0.34,0.42,0.33,0.59,0.43,...,6.000,1.000,15.500,0.775,51.166,10.452,7,0.331320,0.0805,0.1272
8,DK,0.860,0.430,0.48,0.34,0.19,0.24,0.47,0.53,0.31,...,6.000,1.000,15.000,0.750,55.676,12.568,8,0.292774,0.1848,0.1260
9,EE,0.770,0.440,0.62,0.19,0.43,0.50,0.29,0.60,0.40,...,6.000,1.000,19.000,0.950,59.437,24.754,9,0.331410,0.0336,0.1484


In [26]:
# 9. TotalScore_category: categories based on df_totalscore
if 'df_totalscore' in data.columns:
    data['TotalScore_category'] = pd.cut(data['df_totalscore'], bins=[-float('inf'), 5, 10, float('inf')], 
                                         labels=['Low', 'Medium', 'High'], right=False)

# 10. TotalScore_category_encoded: encoding of TotalScore_category
if 'TotalScore_category' in data.columns:
    score_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
    data['TotalScore_category_encoded'] = data['TotalScore_category'].map(score_mapping)

In [27]:
columns_order = [
    'Country', 'Country_ID', 'QB1', 'QB10', 'QB15.1', 'QB15.10', 'QB15.11', 'QB15.12', 'QB15.13',
    'QB15.2', 'QB15.3', 'QB15.4', 'QB15.5', 'QB15.6', 'QB15.7', 'QB15.8', 'QB15.9',
    'QB4.1', 'QB4.2', 'QB4.3', 'QB4T', 'QB5', 'QB6', 'QB12', 'QB13', 'QB2',
    'QB6_QB13', 'QB15.6_QB15.7', 'QB_avg', 'CPI 2020', 'CPI 2021', 'CPI 2022', 'CPI 2023',
    'CPI_avg', 'CPI_2020_minus_avg', 'CPI_2021_minus_avg', 'CPI_2022_minus_avg',
    'CPI_2023_minus_avg', 'CPI_percent_change', 'CPI_category', 'CPI_category_encoded',
    'TotalScore_category', 'TotalScore_category_encoded', 'df_totalscore', 'df_fulfillment',
    'dj_total score', 'dj_fulfillment', 'tindex_total', 'tindex_fulfillment'
]
